# Week 10 (Live) — Multi-Agent Travel Planner (Student)

## 0. Setup

In [ ]:
# HINT: %pip install -q langgraph langchain-openai langchain-tavily python-dotenv requests

# HINT: import os, requests
# HINT: from dotenv import load_dotenv, find_dotenv
# HINT: load_dotenv(find_dotenv(), override=True)

# HINT: write a small _check_key(name) helper that reads os.getenv(name),
#       prints whether it loaded, and returns the value

# HINT: call it for OPENAI_API_KEY, TAVILY_API_KEY, OPENWEATHER_API_KEY,
#       AVIATIONSTACK_API_KEY - store each return value in a variable

# HINT: from langchain_openai import ChatOpenAI
# HINT: create llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## 1. Shared State

In [ ]:
# HINT: from typing import TypedDict
# HINT: class AgentState(TypedDict): define these fields ->
#       query: str            (user's current message)
#       mode: str              ("recommend_destination" or "build_itinerary")
#       destination: str       (chosen destination name)
#       dest_iata: str          (destination airport code)
#       origin_iata: str        (user's departure airport code)
#       num_days: str           (trip length)
#       travelers: str          (who's traveling)
#       weather_data: str       (filled in by Weather Agent)
#       research_data: str      (filled in by Research Agent)
#       flight_data: str        (filled in by Flight Agent)
#       final_output: str       (filled in by Writer Agent)


## 2. The Weather / Destination Tool

In [ ]:
# HINT: from langchain_core.tools import tool

# HINT: create DESTINATION_SHORTLIST - a dict of city name -> {"query": "City,CC",
#       "iata": "XXX"} for ~10-15 destinations spanning both hemispheres

# HINT: write _get_current_weather(city_query) ->
#       - requests.get("https://api.openweathermap.org/data/2.5/weather",
#         params={"q": city_query, "appid": OPENWEATHER_API_KEY, "units": "metric"})
#       - return a dict with "temp" and "description" pulled from the JSON response

# HINT: write an @tool-decorated weather_recommendation_tool(_="") that:
#       - loops over DESTINATION_SHORTLIST, calling _get_current_weather for each
#         (wrap each call in try/except so one failure doesn't break the loop)
#       - sorts results warmest-first
#       - returns the top 5 as a bullet-point string

# HINT: test it - print(weather_recommendation_tool.invoke(""))


## 3. The Research Tool

In [ ]:
# HINT: create MOCK_RESEARCH_FACTS - a short list of generic travel fact strings
#       (used only as a fallback)

# HINT: check TAVILY_API_KEY -> if present, try importing TavilySearch from
#       langchain_tavily and create a client (wrap in try/except ImportError)

# HINT: write an @tool-decorated web_search_tool(query) that:
#       - if a real Tavily client exists, call it and extract each result's
#         "content" field
#       - otherwise fall back to MOCK_RESEARCH_FACTS


## 4. The Flight Tool

In [ ]:
# HINT: write an @tool-decorated flight_search_tool(origin_iata, dest_iata) that:
#       - returns early with a friendly message if AVIATIONSTACK_API_KEY is missing
#       - wraps the actual requests.get(...) call in try/except (network can fail)
#       - calls "http://api.aviationstack.com/v1/flights" with params:
#         access_key, dep_iata=origin_iata, arr_iata=dest_iata, limit=5
#       - checks if the JSON response itself contains an "error" key (Aviationstack
#         returns errors INSIDE a 200 response for things like rate limits)
#       - handles the case of zero flights found (a legitimate result, not an error)
#       - otherwise, formats each flight as a bullet: airline, flight number,
#         departure/arrival times, status


## 5. Supervisor Node

In [ ]:
# HINT: def supervisor_node(state):
#       - print a "Thought" message showing state["mode"]
#       - return {} (it only routes, never does the work)

# HINT: def route_after_supervisor(state) -> str:
#       - if state["mode"] == "recommend_destination": print your decision,
#         return "weather_agent"
#       - else: print your decision, return "research_agent"


## 6. Weather Agent Node

In [ ]:
# HINT: def weather_agent_node(state):
#       - print a "Thought" message
#       - call weather_recommendation_tool.invoke("")
#       - print which tool was called and what it returned
#       - return {"weather_data": <the result>}


## 7. Research Agent Node

In [ ]:
# HINT: def research_agent_node(state):
#       - print a "Thought" message using state["destination"]
#       - build a search string like f"top attractions and things to do in
#         {state['destination']}" and call web_search_tool.invoke(...) with it
#       - print which tool was called and what it returned
#       - return {"research_data": <the result>}


## 8. Flight Agent Node

In [ ]:
# HINT: def flight_agent_node(state):
#       - print a "Thought" message using state["origin_iata"] and state["dest_iata"]
#       - call flight_search_tool.invoke({"origin_iata": ..., "dest_iata": ...})
#         (notice: a dict of TWO arguments this time, not a single string)
#       - print which tool was called and what it returned
#       - return {"flight_data": <the result>}


## 9. Itinerary Writer Agent Node

In [ ]:
# HINT: def writer_agent_node(state):
#       - if state["mode"] == "recommend_destination":
#             build a short prompt asking the LLM to summarize state["weather_data"]
#             as a friendly top-3 recommendation
#       - else:
#             build a richer prompt that includes state["destination"],
#             state["num_days"], state["travelers"], state["research_data"],
#             and state["flight_data"] - and explicitly instruct the LLM to
#             tailor pacing/activities for the travelers described (e.g. rest
#             breaks for senior citizens, family-friendly stops for kids), and
#             to treat the flight info as illustrative only
#       - call llm.invoke(prompt)
#       - print that the LLM produced a final_output
#       - return {"final_output": response.content}


## 10. Wire It Together: Build and Compile the Graph

In [ ]:
# HINT: from langgraph.graph import StateGraph, START, END
# HINT: builder = StateGraph(AgentState)

# HINT: builder.add_node(...) for "supervisor", "weather_agent", "research_agent",
#       "flight_agent", "writer_agent"

# HINT: builder.add_edge(START, "supervisor")

# HINT: builder.add_conditional_edges(
#           "supervisor",
#           route_after_supervisor,
#           {"weather_agent": "weather_agent", "research_agent": "research_agent"},
#       )

# HINT: builder.add_edge("weather_agent", "writer_agent")
# HINT: builder.add_edge("research_agent", "flight_agent")
# HINT: builder.add_edge("flight_agent", "writer_agent")
# HINT: builder.add_edge("writer_agent", END)

# HINT: graph = builder.compile()

# HINT: try/except: print(graph.get_graph().draw_mermaid())


## 11. Run It — Destination Recommendation

In [ ]:
# HINT: build recommend_state matching AgentState:
#       - query = "Where in the world has good summer weather right now?"
#       - mode = "recommend_destination"
#       - every other field = ""

# HINT: result = graph.invoke(recommend_state)
# HINT: print result["final_output"]


## 12. Run It — Full Itinerary

In [ ]:
# HINT: build itinerary_state matching AgentState:
#       - query = something like "Plan my trip to Barcelona"
#       - mode = "build_itinerary"
#       - destination = "Barcelona", dest_iata = DESTINATION_SHORTLIST["Barcelona"]["iata"]
#       - origin_iata = your own example airport code (e.g. "BOM")
#       - num_days = "4", travelers = "2 adults and 1 senior citizen"
#       - the rest = ""

# HINT: result = graph.invoke(itinerary_state)
# HINT: print result["final_output"]


## 13. Interactive Trip Planner (Chatbot with Slot-Filling Memory)

In [ ]:
# HINT: write safe_input(prompt) - same headless-safe try/except pattern as
#       the first notebook's chatbot cell

# HINT: create an empty trip = {} dict - THIS is where real memory across turns
#       lives, outside of AgentState entirely

# HINT: ask the first question with safe_input, e.g. "Where's it summer right now?"
# HINT: build a recommend_state, call graph.invoke(...) ONCE, print the Bot's answer

# HINT: ask which destination the user wants -> match it (case-insensitive)
#       against DESTINATION_SHORTLIST, with a sensible default if unrecognized
# HINT: store trip["destination"] and trip["dest_iata"]

# HINT: ask how many days -> store trip["num_days"] (with a default if blank)
# HINT: ask about kids/senior citizens -> store trip["travelers"] (with a default)
# HINT: ask for the departure airport code -> store trip["origin_iata"] (uppercase it)

# HINT: NOW build one complete full_state using everything in trip,
#       mode="build_itinerary", and call graph.invoke(...) a SECOND time
# HINT: print the final itinerary


## Next Steps (for when we "improvise")

- Add a real Hotel Agent (e.g. via Amadeus) alongside the Flight Agent.
- Filter flights by the user's actual travel date, if a paid API plan supports it.
- Expand or replace `DESTINATION_SHORTLIST` with a less curated data source.
- Let the Weather Agent's options double as `dest_iata` choices automatically.